# Can we predict next-month churn?
Target: `churn_next_month` (1 = account cancels in the month after the snapshot). Data: monthly account snapshots (one row per account-month).

Plan: audit the data (including leakage), build baselines, evaluate with splits that match real use, then summarise.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv')
print(df.shape, 'accounts:', df.account_ref.nunique(), '| missing:', int(df.isna().sum().sum()), '| dup rows:', int(df.duplicated().sum()))
print('churn rate:', round(df.churn_next_month.mean(), 4), '| months', df.snapshot_month.min(), '->', df.snapshot_month.max())
df.head()

## 1. Leakage check: `lifetime_months`
It is the *total* months an account was active, from a CRM export dated 2026-01-05, i.e. after every snapshot. That is information from the future. Check how it relates to the target.

In [ ]:
gap = df.lifetime_months - df.tenure_months
print('Churn rows, lifetime - tenure:'); print(gap[df.churn_next_month == 1].value_counts().head())
print('\nNon-churn rows, lifetime - tenure:'); print(gap[df.churn_next_month == 0].describe().round(2))
print('\nAUC of (tenure - lifetime) alone:', round(roc_auc_score(df.churn_next_month, -gap), 3))

For every churn row `lifetime_months == tenure_months` exactly (the snapshot is the account's last), while non-churners always have lifetime > tenure. The column encodes the answer and would not be known at prediction time. **It is excluded from all models**; any score that used it would be meaninglessly high.

## 2. Exploration
Churn is ~15% overall but varies a lot by calendar month, and accounts appear in many rows.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
df.groupby('snapshot_month').churn_next_month.mean().plot(ax=ax[0], marker='o'); ax[0].set_title('Churn rate by snapshot month'); ax[0].tick_params(axis='x', rotation=90)
df.groupby('tenure_months').churn_next_month.mean().loc[:15].plot(ax=ax[1], marker='o'); ax[1].set_title('Churn rate by tenure (months)')
df.groupby('support_tickets').churn_next_month.mean().plot(ax=ax[2], marker='o'); ax[2].set_title('Churn rate by tickets')
plt.tight_layout(); plt.show()
print(df.groupby('plan').agg(rows=('plan', 'size'), churn=('churn_next_month', 'mean'), fee=('monthly_fee', 'median'), seats=('seats', 'median')).round(3))
print(df.groupby('churn_next_month')[['usage_hours', 'support_tickets', 'tenure_months', 'seats', 'monthly_fee']].mean().round(2))
print('rows per account:', df.groupby('account_ref').size().describe()[['min', '50%', 'max']].to_dict())

Churn spikes in late 2024 / early 2025 (a period effect). Because accounts repeat across rows and conditions shift over time, a random row split would be optimistic. I use (a) account-grouped CV and (b) a forward-in-time holdout.

## 3. Features
Only information available at snapshot time: current-row fields plus the account's *previous* snapshots (lagged usage/tickets). Calendar month is included as a seasonality feature.

In [ ]:
d = df.copy()
d['month'] = pd.PeriodIndex(d.snapshot_month, freq='M')
d = d.sort_values(['account_ref', 'month']).reset_index(drop=True)
d['cal_month'] = d.month.dt.month
d['fee_per_seat'] = d.monthly_fee / d.seats
g = d.groupby('account_ref')
d['m_idx'] = d.month.dt.year * 12 + d.month.dt.month
contig = (d.m_idx - g.m_idx.shift(1)) == 1                                 # previous row is the prior month
d['usage_prev'] = g.usage_hours.shift(1).where(contig)
d['usage_change'] = d.usage_hours - d.usage_prev
d['tickets_prev'] = g.support_tickets.shift(1).where(contig)
d = pd.concat([d, pd.get_dummies(d.plan, prefix='plan', dtype=float)], axis=1)
FEATS = ['seats', 'monthly_fee', 'fee_per_seat', 'tenure_months', 'usage_hours', 'support_tickets', 'cal_month',
         'usage_prev', 'usage_change', 'tickets_prev'] + [c for c in d if c.startswith('plan_')]
y = d.churn_next_month.values; X = d[FEATS]; groups = d.account_ref.values
print(len(FEATS), 'features | lifetime_months used:', 'lifetime_months' in FEATS, '| lag coverage: %.2f' % contig.mean())

## 4. Models
Logistic regression (median-imputed lags) and gradient boosting (native NaN handling). Metrics: ROC-AUC and PR-AUC; a random model's PR-AUC equals the prevalence.

In [ ]:
def logit(): return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
def gbm(): return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=0)
MODELS = {'logistic': logit, 'grad_boost': gbm}

# (a) account-grouped 5-fold CV
oof = {k: np.zeros(len(d)) for k in MODELS}
for tr_i, te_i in GroupKFold(5).split(X, y, groups):
    for k, f in MODELS.items():
        oof[k][te_i] = f().fit(X.iloc[tr_i], y[tr_i]).predict_proba(X.iloc[te_i])[:, 1]
rows = [{'model': k, 'ROC-AUC': roc_auc_score(y, p), 'PR-AUC': average_precision_score(y, p)} for k, p in oof.items()]
print('Grouped CV | random-model PR-AUC = prevalence = %.3f' % y.mean()); display(pd.DataFrame(rows).round(3))

In [ ]:
# (b) forward holdout: train through 2025-04, test 2025-06 onward (1-month gap)
tr = (d.month <= pd.Period('2025-04')).values; te = (d.month >= pd.Period('2025-06')).values
res = []; pt = {}
for k, f in MODELS.items():
    pt[k] = f().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    res.append({'model': k, 'ROC-AUC': roc_auc_score(y[te], pt[k]), 'PR-AUC': average_precision_score(y[te], pt[k])})
print('Forward holdout: train %d, test %d rows, test prevalence %.3f' % (tr.sum(), te.sum(), y[te].mean())); display(pd.DataFrame(res).round(3))

## 5. Drivers and practical usefulness

In [ ]:
m = gbm().fit(X[tr], y[tr])
pi = permutation_importance(m, X[te], y[te], scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=FEATS).sort_values().plot.barh(figsize=(6, 4), title='Permutation importance (drop in holdout AUC)'); plt.show()

p = pt['grad_boost']; yt = y[te]
k = int(0.1 * len(p)); top = np.argsort(-p)[:k]
print('Top decile: churn rate %.3f vs base %.3f (%.1fx lift); captures %.0f%% of churners' % (yt[top].mean(), yt.mean(), yt[top].mean() / yt.mean(), 100 * yt[top].sum() / yt.sum()))
pr, rc, _ = precision_recall_curve(yt, p)
plt.plot(rc, pr); plt.axhline(yt.mean(), ls='--', c='gray'); plt.xlabel('recall'); plt.ylabel('precision'); plt.title('Forward holdout PR curve'); plt.show()

## 6. Conclusion
(Numbers from my run; re-running reproduces them, since seeds are fixed.)

| Evaluation | Model | ROC-AUC | PR-AUC (random = prevalence) |
|---|---|---|---|
| Account-grouped 5-fold CV (prev. 0.154) | logistic | 0.68 | 0.30 |
| | gradient boosting | 0.80 | 0.48 |
| Forward holdout, 2025-06 onward (prev. 0.093) | logistic | 0.68 | 0.19 |
| | gradient boosting | 0.67 | 0.19 |

* **Yes, but only modestly.** There is real signal (tenure, usage, tickets), but on genuinely future months the models reach AUC ~0.67-0.68 and PR-AUC ~0.19 vs 0.09 for chance. The top-scored 10% of accounts churn at ~2x the base rate and capture ~21% of churners.
* **The CV score (0.80) is optimistic.** Random-in-time folds let the model learn the late-2024/early-2025 churn spike from neighbouring months; that does not carry over to the future. Trust the forward holdout.
* **`lifetime_months` must not be used.** It equals `tenure_months` for every churner (AUC 1.0 on its own) because it is derived from a future CRM export. It was excluded.
* **Caveats:** period effects dominate, the holdout is only ~6 months, and rows from the same account are correlated. Next steps: find what drove the late-2024 spike (pricing change? campaign?), add event/billing features, and re-evaluate with rolling-origin backtests before acting on scores.